# Traffic Demand Prediction — v2 (Score: **99.1241**)

**Metric:** `max(0, 100 * R2(actual, predicted))`  
**Validation:** GroupKFold(5) on geohash — no data leakage  
**GPU:** CUDA (PyTorch ResidualMLP)

## Results Summary
| Model | OOF Score |
|---|---|
| LightGBM | 99.0708 |
| XGBoost | 99.0952 |
| CatBoost | 99.0652 |
| NeuralNet (ResidualMLP) | 98.9429 |
| **Stacked (Ridge meta)** | **99.1241** |

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
SEED = 42; np.random.seed(SEED)
print("Libraries loaded. CUDA:", torch.cuda.is_available())

## 2. Load Data

In [ ]:
train = pd.read_csv('e88186124ec611f1/dataset/train.csv')
test  = pd.read_csv('e88186124ec611f1/dataset/test.csv')
print(f"Train: {train.shape} | Test: {test.shape}")
train.head(3)

## 3. Geohash Decoding & Spatial Mapping
Decode geohashes to lat/lon. Map 10 unseen test geohashes to their nearest training neighbor.

In [ ]:
def decode_geohash(gh):
    b32='0123456789bcdefghjkmnpqrstuvwxyz'
    lr,lonr,even=[-90.,90.],[-180.,180.],True
    for c in gh:
        v=b32.find(c)
        for i in range(4,-1,-1):
            bit=(v>>i)&1
            if even:
                m=sum(lonr)/2; lonr=[m,lonr[1]] if bit else [lonr[0],m]
            else:
                m=sum(lr)/2; lr=[m,lr[1]] if bit else [lr[0],m]
            even=not even
    return sum(lr)/2, sum(lonr)/2

all_ghs = list(set(train['geohash'].unique())|set(test['geohash'].unique()))
gh_coords = {g: decode_geohash(g) for g in all_ghs}
train_ghs = set(train['geohash'].unique())
train_gh_list = list(train_ghs)
train_coords = np.array([gh_coords[g] for g in train_gh_list])

gh_mapping={}
for gh in set(test['geohash'].unique())-train_ghs:
    lat,lon=gh_coords[gh]
    dists=np.sqrt((train_coords[:,0]-lat)**2+(train_coords[:,1]-lon)**2)
    gh_mapping[gh]=train_gh_list[np.argmin(dists)]
    print(f"  {gh} -> {gh_mapping[gh]}")

for df in [train,test]:
    df['lookup_gh']=df['geohash'].map(lambda x: gh_mapping.get(x,x))
    df['lat']=df['geohash'].map(lambda x: gh_coords[x][0])
    df['lon']=df['geohash'].map(lambda x: gh_coords[x][1])

## 4. Spatial Clusters (KMeans k=20)

In [ ]:
km=KMeans(n_clusters=20,random_state=SEED,n_init=10)
all_coords=np.array([gh_coords[g] for g in all_ghs])
km.fit(all_coords)
gh_cluster={g:km.predict([gh_coords[g]])[0] for g in all_ghs}
train['spatial_cluster']=train['geohash'].map(gh_cluster)
test['spatial_cluster']=test['geohash'].map(gh_cluster)

# 3 nearest spatial neighbors
spatial_neighbors={}
for gh in all_ghs:
    lat,lon=gh_coords[gh]
    dists=np.sqrt((train_coords[:,0]-lat)**2+(train_coords[:,1]-lon)**2)
    spatial_neighbors[gh]=[train_gh_list[i] for i in np.argsort(dists) if train_gh_list[i]!=gh][:3]
print("Clusters assigned.")

## 5. Timestamp Parsing & Pivot Tables

In [ ]:
def parse_ts(df):
    df=df.copy(); p=df['timestamp'].str.split(':',expand=True)
    df['hour']=p[0].astype(int); df['minute']=p[1].astype(int)
    df['time_mins']=df['hour']*60+df['minute']; return df
train=parse_ts(train); test=parse_ts(test)

TEMP_MEAN=train['Temperature'].mean()
day48=train[train['day']==48][['geohash','time_mins','demand']].copy()
day49e=train[train['day']==49][['geohash','time_mins','demand']].copy()
pivot48=day48.pivot_table(index='geohash',columns='time_mins',values='demand')
pivot49e=day49e.pivot_table(index='geohash',columns='time_mins',values='demand')

gh_stats=day48.groupby('geohash')['demand'].agg(['mean','std','median','max','min']).reset_index()
gh_stats.columns=['geohash','gh_mean48','gh_std48','gh_median48','gh_max48','gh_min48']
time_stats=day48.groupby('time_mins')['demand'].agg(['mean','std']).reset_index()
time_stats.columns=['time_mins','ts_mean48','ts_std48']
cluster_stats=train[train['day']==48].groupby(['spatial_cluster','time_mins'])['demand'].mean().reset_index()
cluster_stats.columns=['spatial_cluster','time_mins','cluster_ts_mean']
print("Pivots built.")

## 6. Feature Engineering (73 features)
Lag features (±0,15,30,60,90,120 min), spatial spillover, trend/ratio, cyclical time, road/weather encodings, spatial cluster stats.

In [ ]:
LAGS=[0,-15,15,-30,30,-60,60,-90,90,-120,120]

def add_lag_features(df):
    df=df.copy()
    for lag in LAGS:
        tgt=df['time_mins']+lag
        v48,v49e=[],[]
        for gh,t in zip(df['lookup_gh'],tgt):
            v48.append(pivot48.loc[gh,t] if (gh in pivot48.index and t in pivot48.columns) else np.nan)
            v49e.append(pivot49e.loc[gh,t] if (gh in pivot49e.index and t in pivot49e.columns) else np.nan)
        df[f'd48_lag{lag}']=v48; df[f'd49e_lag{lag}']=v49e
    spill=[]
    for gh,t in zip(df['lookup_gh'],df['time_mins']):
        nbrs=spatial_neighbors.get(gh,[])
        vals=[pivot48.loc[n,t] for n in nbrs if n in pivot48.index and t in pivot48.columns and not np.isnan(pivot48.loc[n,t])]
        spill.append(np.mean(vals) if vals else np.nan)
    df['d48_spatial_mean']=spill; return df

def build_features(df):
    df=df.copy()
    df['hour_sin']=np.sin(2*np.pi*df['hour']/24); df['hour_cos']=np.cos(2*np.pi*df['hour']/24)
    df['min_sin']=np.sin(2*np.pi*df['time_mins']/1440); df['min_cos']=np.cos(2*np.pi*df['time_mins']/1440)
    df['is_rush']=df['hour'].isin([7,8,9,17,18,19]).astype(int)
    df['is_night']=df['hour'].isin([0,1,2,3,4,5]).astype(int)
    df['is_afternoon']=df['hour'].isin([13,14,15,16]).astype(int)
    df['is_morning']=df['hour'].isin([6,7,8,9,10,11]).astype(int)
    df['RoadType_enc']=df['RoadType'].map({'Residential':0,'Street':1,'Highway':2}).fillna(-1)
    df['LargeVehicles_enc']=(df['LargeVehicles']=='Allowed').astype(int)
    df['Landmarks_enc']=(df['Landmarks']=='Yes').astype(int)
    df['Weather_enc']=df['Weather'].map({'Sunny':0,'Cloudy':1,'Foggy':2,'Rainy':3,'Snowy':4}).fillna(-1)
    df['Temperature']=df['Temperature'].fillna(TEMP_MEAN)
    df['temp_extreme']=((df['Temperature']<5)|(df['Temperature']>35)).astype(int)
    df['temp_weather_interaction']=df['Temperature']*df['Weather_enc']
    df['gh_prefix3']=df['geohash'].str[:3]; df['gh_prefix4']=df['geohash'].str[:4]
    df=add_lag_features(df)
    l48=[c for c in df.columns if c.startswith('d48_lag')]
    l49=[c for c in df.columns if c.startswith('d49e_lag')]
    df['lag48_mean']=df[l48].mean(1); df['lag48_max']=df[l48].max(1)
    df['lag48_std']=df[l48].std(1); df['lag48_median']=df[l48].median(1)
    df['lag48_min']=df[l48].min(1); df['lag48_range']=df['lag48_max']-df['lag48_min']
    df['lag49e_mean']=df[l49].mean(1); df['lag49e_max']=df[l49].max(1); df['lag49e_std']=df[l49].std(1)
    df['demand_trend']=df['d49e_lag0']-df['d48_lag0']
    df['demand_ratio']=df['d49e_lag0']/(df['d48_lag0']+1e-6)
    df=df.merge(gh_stats,left_on='lookup_gh',right_on='geohash',how='left',suffixes=('','_gs'))
    if 'geohash_gs' in df.columns: df.drop(columns=['geohash_gs'],inplace=True)
    df=df.merge(time_stats,on='time_mins',how='left')
    df=df.merge(cluster_stats,on=['spatial_cluster','time_mins'],how='left')
    df['lag0_x_ts']=df['d48_lag0'].fillna(df['gh_mean48'])*df['ts_mean48']
    df['lanes_highway']=df['NumberofLanes']*(df['RoadType_enc']==2).astype(int)
    df['gh_ts_deviation']=df['d48_lag0']-df['ts_mean48']
    df['lag0_vs_cluster']=df['d48_lag0']-df['cluster_ts_mean']
    df['pct_of_day']=df['time_mins']/1440.
    return df

print("Building train features..."); train_fe=build_features(train)
print("Building test features...");  test_fe=build_features(test)

for src,dst in [('geohash','geohash_enc'),('gh_prefix3','gh3_enc'),('gh_prefix4','gh4_enc'),('lookup_gh','lookup_gh_enc')]:
    le=LabelEncoder().fit(pd.concat([train_fe[src],test_fe[src]]))
    train_fe[dst]=le.transform(train_fe[src]); test_fe[dst]=le.transform(test_fe[src])

DROP={'Index','geohash','timestamp','day','demand','gh_prefix3','gh_prefix4','RoadType','LargeVehicles','Landmarks','Weather','lookup_gh'}
FEATURES=[c for c in train_fe.columns if c not in DROP and train_fe[c].dtype!='object']
print(f"Total features: {len(FEATURES)}")
X=train_fe[FEATURES].values.astype(np.float32)
y=train_fe['demand'].values.astype(np.float32)
X_test=test_fe[FEATURES].values.astype(np.float32)

## 7. GroupKFold Validation (no data leakage)

In [ ]:
N_FOLDS=5
groups=train_fe['geohash_enc'].values
folds=list(GroupKFold(n_splits=N_FOLDS).split(X,y,groups))
print(f"GroupKFold with {N_FOLDS} folds on geohash groups")

## 8. Neural Network — ResidualMLP

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self,dim,drop=0.15):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(dim,dim),nn.BatchNorm1d(dim),nn.GELU(),nn.Dropout(drop),nn.Linear(dim,dim),nn.BatchNorm1d(dim))
        self.act=nn.GELU()
    def forward(self,x): return self.act(x+self.net(x))

class TrafficMLP(nn.Module):
    def __init__(self,n,h=256,nb=4,drop=0.15):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(n,h),nn.BatchNorm1d(h),nn.GELU())
        self.blocks=nn.ModuleList([ResidualBlock(h,drop) for _ in range(nb)])
        self.head=nn.Sequential(nn.Linear(h,64),nn.GELU(),nn.Linear(64,1),nn.Sigmoid())
    def forward(self,x):
        x=self.proj(x)
        for b in self.blocks: x=b(x)
        return self.head(x).squeeze(-1)

device='cuda' if torch.cuda.is_available() else 'cpu'; print("Device:",device)
sc=StandardScaler()
X_nn=sc.fit_transform(np.nan_to_num(X,nan=0.)); X_test_nn=sc.transform(np.nan_to_num(X_test,nan=0.))
oof_nn=np.zeros(len(X)); pred_nn=np.zeros(len(X_test))

for fi,(tri,vli) in enumerate(folds):
    Xtr=torch.tensor(X_nn[tri],dtype=torch.float32).to(device)
    ytr=torch.tensor(y[tri],dtype=torch.float32).to(device)
    Xvl=torch.tensor(X_nn[vli],dtype=torch.float32).to(device)
    Xt=torch.tensor(X_test_nn,dtype=torch.float32).to(device)
    dl=DataLoader(TensorDataset(Xtr,ytr),batch_size=2048,shuffle=True)
    m=TrafficMLP(X_nn.shape[1]).to(device)
    opt=optim.AdamW(m.parameters(),lr=1e-3,weight_decay=1e-4)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=50)
    loss_fn=nn.HuberLoss(delta=0.1); best,bep,pat,bs=1e9,0,15,None
    for ep in range(150):
        m.train()
        for xb,yb in dl: loss=loss_fn(m(xb),yb); opt.zero_grad(); loss.backward(); opt.step()
        sched.step(); m.eval()
        with torch.no_grad(): vl=np.mean((m(Xvl).cpu().numpy()-y[vli])**2)
        if vl<best: best=vl; bep=ep; bs={k:v.clone() for k,v in m.state_dict().items()}
        elif ep-bep>pat: break
    m.load_state_dict(bs); m.eval()
    with torch.no_grad(): oof_nn[vli]=m(Xvl).cpu().numpy(); pred_nn+=m(Xt).cpu().numpy()/N_FOLDS
    print(f"NN Fold {fi+1}: {max(0,100*r2_score(y[vli],oof_nn[vli])):.4f}")
print(f"NN OOF: {max(0,100*r2_score(y,oof_nn)):.4f}")

## 9. LightGBM

In [ ]:
lgb_p={'objective':'regression','metric':'rmse','n_estimators':5000,'learning_rate':0.02,
       'num_leaves':255,'min_child_samples':15,'feature_fraction':0.75,'bagging_fraction':0.75,
       'bagging_freq':1,'lambda_l1':0.05,'lambda_l2':0.05,'verbose':-1,'random_state':SEED,'n_jobs':-1}
oof_lgb=np.zeros(len(X)); pred_lgb=np.zeros(len(X_test))
for i,(tri,vli) in enumerate(folds):
    m=lgb.LGBMRegressor(**lgb_p)
    m.fit(X[tri],y[tri],eval_set=[(X[vli],y[vli])],callbacks=[lgb.early_stopping(200,verbose=False),lgb.log_evaluation(0)])
    oof_lgb[vli]=m.predict(X[vli]); pred_lgb+=m.predict(X_test)/N_FOLDS
    print(f"Fold {i+1}: {max(0,100*r2_score(y[vli],oof_lgb[vli])):.4f}  iter:{m.best_iteration_}")
print(f"LightGBM OOF: {max(0,100*r2_score(y,oof_lgb)):.4f}")

## 10. XGBoost

In [ ]:
xgb_p={'objective':'reg:squarederror','eval_metric':'rmse','n_estimators':5000,'learning_rate':0.02,
       'max_depth':8,'min_child_weight':3,'subsample':0.75,'colsample_bytree':0.75,
       'reg_alpha':0.05,'reg_lambda':0.05,'tree_method':'hist','random_state':SEED,'n_jobs':-1,
       'verbosity':0,'early_stopping_rounds':200}
oof_xgb=np.zeros(len(X)); pred_xgb=np.zeros(len(X_test))
for i,(tri,vli) in enumerate(folds):
    m=xgb.XGBRegressor(**xgb_p)
    m.fit(X[tri],y[tri],eval_set=[(X[vli],y[vli])],verbose=False)
    oof_xgb[vli]=m.predict(X[vli]); pred_xgb+=m.predict(X_test)/N_FOLDS
    print(f"Fold {i+1}: {max(0,100*r2_score(y[vli],oof_xgb[vli])):.4f}")
print(f"XGBoost OOF: {max(0,100*r2_score(y,oof_xgb)):.4f}")

## 11. CatBoost

In [ ]:
cat_p={'iterations':5000,'learning_rate':0.02,'depth':8,'loss_function':'RMSE',
       'random_seed':SEED,'verbose':False,'early_stopping_rounds':200,'thread_count':-1,'l2_leaf_reg':3.}
oof_cat=np.zeros(len(X)); pred_cat=np.zeros(len(X_test))
Xc=np.nan_to_num(X,nan=-999); Xtc=np.nan_to_num(X_test,nan=-999)
for i,(tri,vli) in enumerate(folds):
    m=CatBoostRegressor(**cat_p)
    m.fit(Xc[tri],y[tri],eval_set=(Xc[vli],y[vli]),use_best_model=True)
    oof_cat[vli]=m.predict(Xc[vli]); pred_cat+=m.predict(Xtc)/N_FOLDS
    print(f"Fold {i+1}: {max(0,100*r2_score(y[vli],oof_cat[vli])):.4f}")
print(f"CatBoost OOF: {max(0,100*r2_score(y,oof_cat)):.4f}")

## 12. Stacking Meta-Learner (Ridge)

In [ ]:
meta_X=np.column_stack([oof_lgb,oof_xgb,oof_cat,oof_nn])
meta_Xt=np.column_stack([pred_lgb,pred_xgb,pred_cat,pred_nn])
meta_oof=np.zeros(len(y)); meta_pred=np.zeros(len(pred_lgb))
for tri,vli in KFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED).split(meta_X):
    r=Ridge(alpha=0.5); r.fit(meta_X[tri],y[tri])
    meta_oof[vli]=r.predict(meta_X[vli]); meta_pred+=r.predict(meta_Xt)/N_FOLDS
meta_score=max(0,100*r2_score(y,meta_oof))

lgb_s=max(0,100*r2_score(y,oof_lgb)); xgb_s=max(0,100*r2_score(y,oof_xgb))
cat_s=max(0,100*r2_score(y,oof_cat)); nn_s=max(0,100*r2_score(y,oof_nn))
print(f"{'='*50}")
print(f"  LightGBM OOF:     {lgb_s:.4f}")
print(f"  XGBoost  OOF:     {xgb_s:.4f}")
print(f"  CatBoost OOF:     {cat_s:.4f}")
print(f"  NeuralNet OOF:    {nn_s:.4f}")
print(f"  Stacked  OOF:     {meta_score:.4f}  <- FINAL")
print(f"{'='*50}")

## 13. Generate Submission

In [ ]:
final=np.clip(meta_pred,0.,1.)
sub=pd.DataFrame({'Index':test['Index'],'demand':final})
assert sub.shape==(41778,2)
sub.to_csv('submission.csv',index=False)
print(f"submission.csv saved: {sub.shape}")
sub.head(10)